# gnafr — interactive R demo

This notebook runs the **gnafr** Australian address-matching workflow end to end using a small in-memory DuckDB database — no G-NAF download required.

Run each cell with **Shift+Enter** (or the ▶ play button) like any Jupyter notebook.

In [ ]:
# Load the package and data.table
library(gnafr)
library(data.table)

## 1. Create an in-memory database

In [ ]:
# ':memory:' creates a DuckDB database that lives only for this session
con <- gnaf_connect(":memory:")
gnaf_init(con)

## 2. Add a few sample addresses

In [ ]:
demo_addresses <- data.table(
  address_label = c(
    "10 MUSGRAVE ROAD, RED HILL QLD 4059",
    "120 MUSGRAVE ROAD, RED HILL QLD 4059",
    "18-20 DRIFT CLOSE, GOLDSBOROUGH QLD 4865"
  ),
  number_first  = c(10L, 120L, 18L),
  number_last   = c(NA_integer_, NA_integer_, 20L),
  street_name   = c("MUSGRAVE", "MUSGRAVE", "DRIFT"),
  street_type   = c("ROAD", "ROAD", "CLOSE"),
  locality_name = c("RED HILL", "RED HILL", "GOLDSBOROUGH"),
  state         = c("QLD", "QLD", "QLD"),
  postcode      = c(4059L, 4059L, 4865L),
  longitude     = c(153.0066, 153.0080, 145.6203),
  latitude      = c(-27.4570, -27.4575, -17.2806)
)

gnaf_add(con, demo_addresses)
gnaf_status(con)

## 3. Match a messy input string

In [ ]:
result <- gnaf_match(
  "unit 5 18-20 drift cl goldsborough 4865",
  con,
  max_results = 2,
  verbose = FALSE
)

In [ ]:
# Inspect the key result columns
result[, .(input_raw, matched, total_score, address_label,
           score_postcode, score_street_name, score_number)]

## Clean up

In [ ]:
gnaf_disconnect(con)